# 03 — Detector de anomalías con threshold dinámico

Núcleo del proyecto. Entrenamos dos detectores **solo con tráfico Normal** (no supervisado) y comparamos:

1. **Isolation Forest** — rápido, interpretable, baseline robusto.
2. **Autoencoder (PyTorch)** — aprende la "forma" del tráfico normal y usa el error de reconstrucción como score.

Trabajamos solo con las 7 features derivables de OpenFlow (ver `docs/INTERFACE.md`).

El **umbral dinámico** se define como un percentil del score sobre una ventana deslizante de polls recientes. Comparamos con un umbral fijo (percentil de la validación) para mostrar la ventaja.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

sys.path.insert(0, str(Path.cwd().parent))
from src.data import get_insdn_path
from src.features import clean_insdn, INSDN_TO_OPENFLOW

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 1. Carga y preparación

Construimos `X` con las 7 features deployable y `y_binary` con 0 = Normal, 1 = cualquier ataque.

In [ ]:
csv = next(get_insdn_path().rglob('*.csv'))
df = pd.read_csv(csv, low_memory=False)
df.columns = df.columns.str.strip()
df = clean_insdn(df)

X = df[list(INSDN_TO_OPENFLOW.keys())].copy()
X['Flow Duration'] = X['Flow Duration'] / 1e6  # µs -> s
X = X.rename(columns=INSDN_TO_OPENFLOW)
y_binary = (df['Label'] != 'Normal').astype(int).values
y_multi = df['Label'].values

print(f'X shape: {X.shape}')
print(f'Normal: {(y_binary == 0).sum()}  |  Ataque: {(y_binary == 1).sum()}')
X.describe()

## 2. Split: entrenamiento solo con Normal

- **`X_train`**: 70% del tráfico Normal → entrena AE y IF.
- **`X_val_normal`**: 30% restante de Normal → ajustar umbral (no se ve durante el fit).
- **`X_test`**: todos los ataques + esos mismos `X_val_normal` → evaluación final binaria.

In [ ]:
is_normal = y_binary == 0
X_normal = X[is_normal]
X_attack = X[~is_normal]
y_attack_multi = y_multi[~is_normal]

X_train, X_val_normal = train_test_split(
    X_normal, test_size=0.3, random_state=RANDOM_STATE
)

X_test = pd.concat([X_val_normal, X_attack], ignore_index=True)
y_test = np.concatenate([
    np.zeros(len(X_val_normal), dtype=int),
    np.ones(len(X_attack), dtype=int),
])
y_test_multi = np.concatenate([
    np.array(['Normal'] * len(X_val_normal)),
    y_attack_multi,
])

print(f'Train (solo Normal): {X_train.shape}')
print(f'Test:                {X_test.shape}  '
      f'(Normal={(y_test==0).sum()}, Ataque={(y_test==1).sum()})')

In [ ]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val_normal)
X_test_s = scaler.transform(X_test)

## 3. Isolation Forest

Entrena con `contamination=0.0` (asumimos que el train es puro Normal). Devolvemos el score como `-score_samples` → mayor = más anómalo.

In [ ]:
iforest = IsolationForest(
    n_estimators=200,
    contamination='auto',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
iforest.fit(X_train_s)
scores_if_val = -iforest.score_samples(X_val_s)
scores_if_test = -iforest.score_samples(X_test_s)
print(f'IF score (val Normal): mean={scores_if_val.mean():.4f}, p95={np.percentile(scores_if_val, 95):.4f}, p99={np.percentile(scores_if_val, 99):.4f}')

## 4. Autoencoder

Arquitectura simple: `7 → 16 → 8 → 4 → 8 → 16 → 7`. Loss MSE. Adam.

Score de anomalía = error de reconstrucción por muestra.

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 4),
        )
        self.decoder = nn.Sequential(
            nn.Linear(4, 8), nn.ReLU(),
            nn.Linear(8, 16), nn.ReLU(),
            nn.Linear(16, in_dim),
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def reconstruction_error(model, X_np, batch=4096):
    model.eval()
    errs = []
    with torch.no_grad():
        for i in range(0, len(X_np), batch):
            xb = torch.tensor(X_np[i:i+batch], dtype=torch.float32, device=DEVICE)
            err = ((model(xb) - xb) ** 2).mean(dim=1)
            errs.append(err.cpu().numpy())
    return np.concatenate(errs)

In [ ]:
ae = Autoencoder(in_dim=X_train_s.shape[1]).to(DEVICE)
opt = torch.optim.Adam(ae.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=DEVICE)
EPOCHS = 30
BATCH = 512
history = []

for epoch in range(EPOCHS):
    ae.train()
    perm = torch.randperm(len(X_train_t))
    epoch_loss = 0.0
    for i in range(0, len(perm), BATCH):
        idx = perm[i:i+BATCH]
        xb = X_train_t[idx]
        opt.zero_grad()
        loss = loss_fn(ae(xb), xb)
        loss.backward(); opt.step()
        epoch_loss += loss.item() * len(idx)
    epoch_loss /= len(perm)
    history.append(epoch_loss)
    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d}  loss={epoch_loss:.5f}')

plt.plot(history); plt.xlabel('epoch'); plt.ylabel('MSE'); plt.title('Curva de entrenamiento AE'); plt.show()

In [ ]:
scores_ae_val = reconstruction_error(ae, X_val_s)
scores_ae_test = reconstruction_error(ae, X_test_s)
print(f'AE score (val Normal): mean={scores_ae_val.mean():.5f}, p95={np.percentile(scores_ae_val, 95):.5f}, p99={np.percentile(scores_ae_val, 99):.5f}')

## 5. Comparativa de scores

Distribución de scores para Normal vs ataque. Si los dos detectores funcionan, deberíamos ver poco solapamiento.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, scores, name in zip(axes, [scores_if_test, scores_ae_test], ['Isolation Forest', 'Autoencoder']):
    sns.histplot(scores[y_test == 0], bins=80, ax=ax, label='Normal', stat='density', color='C0', alpha=0.6)
    sns.histplot(scores[y_test == 1], bins=80, ax=ax, label='Ataque', stat='density', color='C3', alpha=0.6)
    ax.set_xlim(np.percentile(scores, 0.5), np.percentile(scores, 99.5))
    ax.set_title(name); ax.set_xlabel('score de anomalía'); ax.legend()
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for scores, name in [(scores_if_test, 'IF'), (scores_ae_test, 'AE')]:
    fpr, tpr, _ = roc_curve(y_test, scores)
    auc = roc_auc_score(y_test, scores)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.4f})')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC')
ax.legend(); plt.tight_layout()

## 6. Umbral fijo (baseline) vs umbral dinámico

### Umbral fijo

Usamos el percentil P de los scores sobre la validación Normal. Cualquier muestra con score > umbral se marca como anomalía.

In [ ]:
def evaluate_with_threshold(scores, y_true, thr, name):
    y_pred = (scores > thr).astype(int)
    print(f'\n=== {name}  (thr={thr:.5f}) ===')
    print(classification_report(y_true, y_pred, target_names=['Normal', 'Ataque'], digits=4))
    return y_pred

P = 99  # percentil sobre val Normal
thr_if = np.percentile(scores_if_val, P)
thr_ae = np.percentile(scores_ae_val, P)
_ = evaluate_with_threshold(scores_if_test, y_test, thr_if, f'IF, umbral fijo P{P}')
_ = evaluate_with_threshold(scores_ae_test, y_test, thr_ae, f'AE, umbral fijo P{P}')

### Umbral dinámico

En producción no tenemos un percentil precomputado: el detector ve un stream de polls (cada ~2s) y debe adaptarse al ruido de fondo cambiante.

**Mecanismo:** ventana deslizante de los últimos `W` scores; umbral = percentil `P` de esa ventana × factor de seguridad `k`. Esto evita falsos positivos cuando el tráfico legítimo aumenta gradualmente, pero hay que protegerlo contra un ataque sostenido (la ventana se contamina). Lo hacemos con un mínimo absoluto basado en la calibración inicial.

Aquí lo simulamos sobre el test mezclando muestras en orden aleatorio (como aproximación de un stream).

In [ ]:
def dynamic_threshold_eval(scores, y_true, window=2000, pct=99, k=1.0, min_thr=None):
    """Simula un stream: para cada muestra, umbral = percentil de la ventana anterior.

    `min_thr` actúa como suelo (típicamente el percentil de la calibración inicial)
    para que un ataque sostenido no eleve el umbral hasta esconderse.
    """
    rng = np.random.default_rng(RANDOM_STATE)
    order = rng.permutation(len(scores))
    s = scores[order]
    y = y_true[order]
    y_pred = np.zeros_like(y)
    thresholds = np.zeros_like(s)
    for i in range(len(s)):
        lo = max(0, i - window)
        if i < 50:
            thr = min_thr if min_thr is not None else np.inf
        else:
            thr = np.percentile(s[lo:i], pct) * k
            if min_thr is not None:
                thr = max(thr, min_thr)
        thresholds[i] = thr
        y_pred[i] = int(s[i] > thr)
    return y[order.argsort()], y_pred[order.argsort()], thresholds, order

# Calibrar suelo con percentil de val Normal
for scores, name, thr_min in [
    (scores_if_test, 'IF', np.percentile(scores_if_val, 99)),
    (scores_ae_test, 'AE', np.percentile(scores_ae_val, 99)),
]:
    _, y_pred, _, _ = dynamic_threshold_eval(scores, y_test, window=2000, pct=99, k=1.0, min_thr=thr_min)
    print(f'\n=== {name}, umbral dinámico (window=2000, P99, min=val_P99) ===')
    print(classification_report(y_test, y_pred, target_names=['Normal', 'Ataque'], digits=4))

## 7. Persistencia

Guardamos los modelos y el scaler para que `src/detector.py` los pueda cargar y usar desde la Ryu app.

In [ ]:
import joblib
MODELS = Path('../models')
MODELS.mkdir(exist_ok=True)

joblib.dump(scaler, MODELS / 'scaler.pkl')
joblib.dump(iforest, MODELS / 'iforest.pkl')
torch.save(ae.state_dict(), MODELS / 'autoencoder.pt')
joblib.dump({
    'features': list(X.columns),
    'iforest_thr_p99': float(np.percentile(scores_if_val, 99)),
    'ae_thr_p99': float(np.percentile(scores_ae_val, 99)),
    'ae_arch': {'in_dim': X.shape[1], 'hidden': [16, 8, 4]},
}, MODELS / 'detector_meta.pkl')
print('Modelos guardados en', MODELS.resolve())

## Próximos pasos

- `04_classifier.ipynb`: XGBoost que toma las muestras marcadas como anómalas y clasifica el **tipo** de ataque.
- `src/detector.py`: empaquetar load+predict para que la Ryu app pueda importar.
- Validación en Mininet: ataques reales con `hping3`, tráfico legítimo con `iperf3`.